In [ ]:
# %% [CELL 1] Configuration  <- the only cell you edit (MODE, SMOKE)
# =============================================================================
# TRANSLATION PRE-STEP — English names for every non-Latin S2/S3 record
# Amazon ML Challenge 2026 — Business Entity Resolution
# =============================================================================
# Model: IndicTrans2 Indic->English distilled 200M (AI4Bharat), MIT licence.
# Translates each DISTINCT non-Latin business name once (names are sorted by
# length so batches carry little padding) and writes
#     /kaggle/working/translations_<mode>.parquet
#         entity_id | lang | translation
# Used by the L8 probe, Phase C (blocking) and Stage 2 v3 (features).
# S1 names are always Latin (verified), so only S2/S3 are translated.
#
# RUN      SMOKE = True first (~5 min: 2,000 names, speed + projection),
#          then SMOKE = False. Progress is saved in chunks: an interrupted
#          interactive run resumes where it stopped.
# SETTINGS GPU T4 x1, Internet ON, Kaggle secret HF_TOKEN attached
#          (Add-ons -> Secrets), Persistence Files.
# INPUTS   the competition dataset only.
# =============================================================================

MODE = "test"                 # "test" (teammate C) or "train" (teammate D)
SMOKE = True
SMOKE_NAMES = 2_000

DATA_ROOT = "/kaggle/input/datasets/mahimasunilbachhav/amazon-ml-challenge/student_resource/dataset"
WORK = "/kaggle/working"
MODEL = "ai4bharat/indictrans2-indic-en-dist-200M"
BATCH = 256                   # names per GPU batch (names are short)
NUM_BEAMS = 1                 # greedy decoding: the probe showed 99% exact-enough output
MAX_LEN = 64
CHUNK = 20_000                # names per saved progress chunk

In [ ]:
# %% [CELL 2] Install, import, compatibility fix
import importlib, os, re, subprocess, sys, time, types
from pathlib import Path
import numpy as np
import pandas as pd

def pip(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

def import_ip():
    try:
        from IndicTransToolkit.processor import IndicProcessor
    except ImportError:
        from IndicTransToolkit import IndicProcessor
    return IndicProcessor

try:
    IndicProcessor = import_ip()
except ImportError:
    pip("IndicTransToolkit")
    try:
        IndicProcessor = import_ip()
    except Exception as e:
        IndicProcessor = None
        print(f"IndicTransToolkit unavailable ({e}); using plain language tags.")

import torch
import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
print("transformers", transformers.__version__, "| IndicProcessor:", IndicProcessor is not None)
try:                                              # IndicTrans2 code imports ONNX-export helpers
    importlib.import_module("transformers.onnx")  # that newer transformers removed; inference
except ModuleNotFoundError:                       # never uses them, so empty stand-ins suffice.
    onnx = types.ModuleType("transformers.onnx")
    class OnnxConfig:
        default_fixed_batch, default_fixed_sequence = 2, 8
        def __init__(self, *a, **k): pass
    class OnnxConfigWithPast(OnnxConfig): pass
    class OnnxSeq2SeqConfigWithPast(OnnxConfigWithPast): pass
    onnx.OnnxConfig, onnx.OnnxConfigWithPast = OnnxConfig, OnnxConfigWithPast
    onnx.OnnxSeq2SeqConfigWithPast = OnnxSeq2SeqConfigWithPast
    ou = types.ModuleType("transformers.onnx.utils")
    ou.compute_effective_axis_dimension = lambda d, f, n=0: f if d <= 0 else d
    onnx.utils = ou
    sys.modules["transformers.onnx"], sys.modules["transformers.onnx.utils"] = onnx, ou
    transformers.onnx = onnx
    print("installed stand-in transformers.onnx (export helpers only)")

assert MODE in ("test", "train")
DATA_ROOT, WORK = Path(DATA_ROOT), Path(WORK)
T0 = time.time()
log = lambda m: print(f"  [{(time.time()-T0)/60:5.1f} min] {m}", flush=True)

In [ ]:
# %% [CELL 3] Collect the distinct non-Latin names of the S2/S3 pool
LANG_OF_BLOCK = {0x0900: "hin_Deva", 0x0980: "ben_Beng", 0x0A00: "pan_Guru", 0x0A80: "guj_Gujr",
                 0x0B00: "ory_Orya", 0x0B80: "tam_Taml", 0x0C00: "tel_Telu", 0x0C80: "kan_Knda",
                 0x0D00: "mal_Mlym"}

def lang_of(s):
    for ch in s:
        o = ord(ch)
        if 0x0900 <= o < 0x0D80:
            return LANG_OF_BLOCK[o & ~0x7F]
    return None

COLS = ["entity_id", "business_name", "business_address", "country"]
recs = pd.concat([pd.read_csv(DATA_ROOT / MODE / f"{MODE}_source{i}.tsv", sep="\t", dtype=str,
                              names=COLS, header=0, keep_default_na=False, usecols=[0, 1])
                  for i in (2, 3)], ignore_index=True)
nl = recs[recs["business_name"].str.contains("[\u0900-\u0D7F]", regex=True)].copy()
nl["lang"] = [lang_of(s) for s in nl["business_name"]]
uniq = (nl[["business_name", "lang"]].drop_duplicates("business_name")
        .assign(n=lambda d: d["business_name"].str.len())
        .sort_values(["lang", "n"]).reset_index(drop=True))
log(f"{len(recs):,} S2/S3 records, {len(nl):,} with non-Latin names, {len(uniq):,} distinct names")
print("  distinct names per script:", ", ".join(f"{k} {v:,}" for k, v in uniq["lang"].value_counts().items()))
if SMOKE:
    uniq = uniq.sample(min(SMOKE_NAMES, len(uniq)), random_state=0).sort_values(["lang", "n"]).reset_index(drop=True)
    log(f"SMOKE: translating {len(uniq):,} names")

In [ ]:
# %% [CELL 4] Load IndicTrans2 (needs the HF_TOKEN secret)
assert torch.cuda.is_available(), "GPU not enabled: set Accelerator to GPU T4"
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
assert HF_TOKEN, "No Hugging Face token: add a Kaggle secret HF_TOKEN (Add-ons -> Secrets), ticked"
try:
    from huggingface_hub import model_info
    print("model card licence:", (model_info(MODEL, token=HF_TOKEN).card_data or {}).get("license"))
except Exception as e:
    print("could not read the model card:", e)
tok = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True, token=HF_TOKEN)
# Newer transformers call model.tie_weights(missing_keys=..., ...), but IndicTrans2's
# model code defines tie_weights(self). Load its class directly and let tie_weights
# accept (and ignore) the extra arguments; the weights are tied exactly as before.
from transformers.dynamic_module_utils import get_class_from_dynamic_module
IT2 = get_class_from_dynamic_module("modeling_indictrans.IndicTransForConditionalGeneration",
                                    MODEL, token=HF_TOKEN)
if not getattr(IT2, "_tie_patched", False):
    _orig_tie = IT2.tie_weights
    def _tie_weights(self, *args, **kwargs):
        return _orig_tie(self)
    IT2.tie_weights = _tie_weights
    IT2._tie_patched = True
mdl = IT2.from_pretrained(MODEL, token=HF_TOKEN, torch_dtype=torch.float16).to("cuda").eval()
ip = IndicProcessor(inference=True) if IndicProcessor is not None else None
log(f"model loaded: {sum(p.numel() for p in mdl.parameters())/1e6:.0f}M parameters")


def translate(texts, lang):
    """Translate names from `lang` to English, in batches."""
    out = []
    for i in range(0, len(texts), BATCH):
        batch = texts[i:i + BATCH]
        batch = (ip.preprocess_batch(batch, src_lang=lang, tgt_lang="eng_Latn") if ip is not None
                 else [f"{lang} eng_Latn {x}" for x in batch])
        enc = tok(batch, truncation=True, padding="longest", max_length=MAX_LEN,
                  return_tensors="pt").to("cuda")
        with torch.no_grad():
            try:
                gen = mdl.generate(**enc, num_beams=NUM_BEAMS, max_length=MAX_LEN, use_cache=True)
            except Exception:
                gen = mdl.generate(**enc, num_beams=NUM_BEAMS, max_length=MAX_LEN, use_cache=False)
        dec = tok.batch_decode(gen, skip_special_tokens=True, clean_up_tokenization_spaces=True)
        out += ip.postprocess_batch(dec, lang="eng_Latn") if ip is not None else dec
    return out

In [ ]:
# %% [CELL 5] Translate (progress saved per chunk; an interrupted run resumes)
PARTS = WORK / f"_translate_parts_{MODE}{'_smoke' if SMOKE else ''}"
PARTS.mkdir(parents=True, exist_ok=True)
t_start, done_now = time.time(), 0
for lang, g in uniq.groupby("lang", sort=True):
    names = g["business_name"].tolist()
    for c0 in range(0, len(names), CHUNK):
        part = PARTS / f"{lang}_{c0:08d}.parquet"
        if part.exists():
            continue
        chunk = names[c0:c0 + CHUNK]
        t = time.time()
        pd.DataFrame({"business_name": chunk, "translation": translate(chunk, lang)}).to_parquet(part, index=False)
        done_now += len(chunk)
        rate = done_now / (time.time() - t_start)
        left = len(uniq) - sum(len(pd.read_parquet(p, columns=["business_name"])) for p in PARTS.glob("*.parquet"))
        log(f"{lang}: {c0 + len(chunk):,}/{len(names):,}  ({len(chunk)/(time.time()-t):.0f} names/s)  "
            f"~{left/max(rate,1e-9)/60:.0f} min left overall")
rate = done_now / max(time.time() - t_start, 1e-9)

In [ ]:
# %% [CELL 6] Write translations_<mode>.parquet + checks
tr = pd.concat([pd.read_parquet(p) for p in sorted(PARTS.glob("*.parquet"))], ignore_index=True)
out = nl.merge(tr, on="business_name", how="inner")[["entity_id", "lang", "business_name", "translation"]]
dest = WORK / f"translations_{MODE}{'_smoke' if SMOKE else ''}.parquet"
out.to_parquet(dest, index=False)
empty = (out["translation"].str.strip() == "").mean() * 100
still = out["translation"].str.contains("[\u0900-\u0D7F]", regex=True).mean() * 100
print("\n" + "=" * 72)
print(f"TRANSLATION {MODE.upper()} {'SMOKE' if SMOKE else 'FULL'}: {len(out):,} records "
      f"({len(tr):,} distinct names)  ->  {dest}")
print("=" * 72)
print(f"  speed this run      : {rate:.0f} names/s")
print(f"  empty translations  : {empty:.2f}%     still containing Indian script: {still:.2f}%")
for _, r in out.sample(min(12, len(out)), random_state=1).iterrows():
    print(f"   [{r.lang}] {r.business_name}  ->  {r.translation}")
if SMOKE:
    full_distinct = len(nl.drop_duplicates("business_name"))
    print(f"\n  PROJECTION: {full_distinct:,} distinct names at {rate:.0f} names/s = "
          f"~{full_distinct/max(rate,1e-9)/60:.0f} min (+ ~2 min model load)")
    print("  Smoke run passed. Set SMOKE = False and run again (Save & Run All or interactive).")
else:
    import shutil
    shutil.rmtree(PARTS, ignore_errors=True)       # keep only the final table in the output
    print(f"\n  DONE. Share this notebook's output (or download {dest.name}) with the next step.")
log("finished")